In [ ]:
# --- Parameters (papermill-injected) ---
# The disarmed handheld run (motors OFF -> vibration-free baseline) and the
# armed powered flight (motors ON -> vibration treatment). See coordinator #42.
handheld_fixture = "/mnt/s/flights/rekon10/260705-handheld-noarm/wave-20260705-112443.bin"
handheld_fc      = "/mnt/s/flights/rekon10/260705-handheld-noarm/1980-01-06 10-14-19.bin"
armed_fixture    = "/mnt/s/flights/rekon10/260705-vio-logged/wave-20260705-114708.bin"
armed_fc         = "/mnt/s/flights/rekon10/260705-vio-logged/1980-01-06 10-45-23.bin"
repo_analysis    = "/home/symmetry/coordinator/analysis"   # for ardupilot_log.parse_log
debug = False


# VIO direct-input alignment & comparison — rekon10 260705 pair

**Purpose (coordinator [#42](https://github.com/symmatree/coordinator/issues/42)):** take the two matched
captures and, working only from the *direct input signals*, (1) align the coordinator `vio-ipc-record`
fixture to the FC dataflash log on a common timeline, (2) compare the OAK-D camera IMU against the FC IMU,
(3) ask whether the OAK-D IMU **sees the motor vibration** that the armed flight put on the FC IMU, and
(4) lay out the plan to *post-hoc regenerate VINS pose* from the captured inputs.

| run | fixture | FC log | motors |
|-----|---------|--------|--------|
| **handheld** (baseline) | `chobits_imu`+`chobits_features` | `LOG_DISARMED` | **off** — vibration-free |
| **armed** (treatment)   | `chobits_imu`+`chobits_features` | powered flight | **on** — hard-mounted OAK-D |

**Clocks.** Both FC `.bin`s are 1980-dated (RTC unset at log open), so wall-clock is useless. The fixture
frame carries `t_mono` (recorder `CLOCK_MONOTONIC`) and each `chobits_imu` packet carries the OAK-D device
timestamp `t_dev`. Alignment to the FC log is by **motion cross-correlation**, not timestamps.

**Fixture wire format** (`<ddHI>` frame = `t_mono, t_unix, socket_id, len`, then `len` raw bytes):
- `chobits_imu` (socket 0): **7 doubles** `[t_dev, ax, ay, az, gx, gy, gz]`, calibrated, ROS frame (56 B).
- `chobits_features` (socket 1): `[count, features_ts]` then `count` × **13 doubles**/feature.
  > ⚠️ The `~/vio/oak_d_vins_cpp/feature_tracker.cpp` source packs **14** doubles/feature (trailing
  > `depth = f*baseline/disp`). The *deployed* build in these fixtures packs **13** — verified from the
  > payload length `(len-16)/count = 104 = 13×8`. The replayer (below) re-sends raw bytes, so this only
  > matters for decoding here; flagged so nobody decodes with the wrong stride.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import struct, json, sys, os
from pathlib import Path
from IPython.display import display as ipy_display, Image as ipy_image
import io

sys.path.insert(0, repo_analysis)
from ardupilot_log import parse_log            # canonical FC-log parser

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    ipy_display(ipy_image(data=buf.getvalue()))

def welch_psd(x, fs, nperseg=2048):
    """One-sided Welch PSD, Hann window, 50% overlap (numpy-only, ~scipy.welch)."""
    x = np.asarray(x, float)
    nperseg = int(min(nperseg, len(x)))
    if nperseg < 8:
        return np.array([0.0]), np.array([0.0])
    win = np.hanning(nperseg)
    scale = 1.0 / (fs * (win ** 2).sum())
    step = max(nperseg // 2, 1)
    segs = []
    for s in range(0, len(x) - nperseg + 1, step):
        sp = np.fft.rfft(x[s:s + nperseg] * win)
        p = (np.abs(sp) ** 2) * scale
        p[1:-1] *= 2.0
        segs.append(p)
    return np.fft.rfftfreq(nperseg, 1 / fs), np.mean(segs, axis=0)


In [ ]:
# --- vio-ipc-record fixture decoder ---
_FRAME = struct.Struct("<ddHI")   # t_mono, t_unix, socket_id, length
_IMU   = struct.Struct("<7d")     # t_dev, ax,ay,az, gx,gy,gz

def decode_fixture(path):
    """Decode a vio-ipc-record .bin -> (imu_df, feat_df, manifest).

    imu_df  : t_mono, t_unix, t_dev, ax,ay,az, gx,gy,gz   (accel m/s^2, gyro rad/s, ROS frame)
    feat_df : t_mono, t_unix, features_ts, n_feat, stride (stride should be 13 on deployed build)
    """
    manifest = json.loads(Path(path + ".json").read_text())
    data = Path(path).read_bytes()
    n = len(data); off = 0
    imu_rows, feat_rows, bad = [], [], 0
    while off + _FRAME.size <= n:
        t_mono, t_unix, sid, ln = _FRAME.unpack_from(data, off); off += _FRAME.size
        if off + ln > n:
            bad += 1; break
        p = data[off:off + ln]; off += ln
        if sid == 0:                                   # chobits_imu
            if ln != 56:
                bad += 1; continue
            t_dev, ax, ay, az, gx, gy, gz = _IMU.unpack(p)
            imu_rows.append((t_mono, t_unix, t_dev, ax, ay, az, gx, gy, gz))
        elif sid == 1:                                 # chobits_features
            c   = struct.unpack_from("<d", p, 0)[0]
            fts = struct.unpack_from("<d", p, 8)[0]
            nf  = int(c)
            stride = (ln - 16) // 8 // nf if nf else 0
            feat_rows.append((t_mono, t_unix, fts, nf, stride))
    imu = pd.DataFrame(imu_rows,
        columns=["t_mono","t_unix","t_dev","ax","ay","az","gx","gy","gz"])
    feat = pd.DataFrame(feat_rows,
        columns=["t_mono","t_unix","features_ts","n_feat","stride"])
    if bad:
        print(f"  note: {bad} unparseable/short frame(s) in {Path(path).name}")
    return imu, feat, manifest


In [ ]:
runs = {}
for name, fx in [("handheld", handheld_fixture), ("armed", armed_fixture)]:
    imu, feat, man = decode_fixture(fx)
    imu["a_mag"] = np.linalg.norm(imu[["ax","ay","az"]].values, axis=1)
    imu["g_mag"] = np.linalg.norm(imu[["gx","gy","gz"]].values, axis=1)
    # OAK-D IMU sample rate from the device clock (true spacing, for FFT)
    dt_dev = np.median(np.diff(imu["t_dev"]))
    fs_oak = 1.0 / dt_dev
    runs[name] = dict(imu=imu, feat=feat, man=man, fs_oak=fs_oak)
    strides = feat["stride"].unique()
    print(f"[{name}] IMU {len(imu):>6} pkts @ {fs_oak:5.1f} Hz  "
          f"|a|={imu.a_mag.mean():4.2f}  feats {len(feat):>5} "
          f"(mean {feat.n_feat.mean():4.1f}/frame, stride {strides})")
    assert set(strides) <= {13}, f"unexpected feature stride {strides} (expected 13 on deployed build)"


In [ ]:
# --- FC dataflash logs (IMU0 for motion-alignment; VIBE/GPS/XKF1 for context) ---
def load_fc(path):
    F, dur, parms = parse_log(path, ["IMU","VIBE","GPS","XKF1","RCOU"])
    imu0 = F["IMU"][F["IMU"]["I"] == 0].reset_index(drop=True)
    imu0["g_mag"] = np.linalg.norm(imu0[["GyrX","GyrY","GyrZ"]].values, axis=1)
    imu0["a_mag"] = np.linalg.norm(imu0[["AccX","AccY","AccZ"]].values, axis=1)
    fs = 1.0 / np.median(np.diff(imu0["t_s"]))
    return dict(F=F, imu0=imu0, dur=dur, fs=fs)

runs["handheld"]["fc"] = load_fc(handheld_fc)
runs["armed"]["fc"]    = load_fc(armed_fc)
for name in runs:
    fc = runs[name]["fc"]
    print(f"[{name}] FC log {fc['dur']:6.1f}s  IMU0 logged @ {fc['fs']:4.1f} Hz  "
          f"({len(fc['imu0'])} rows)")
print("\nNyquist note: FC IMU logged @ ~25 Hz (batch-IMU off per #42) and OAK-D IMU @ ~100 Hz. "
      "Neither directly resolves a ~150-250 Hz motor fundamental; the vibration signature to look "
      "for is elevated broadband / aliased energy in armed vs handheld, plus the FC VIBE metric.")


## 1. Align fixture ↔ FC log by motion cross-correlation

Both the OAK-D gyro and the FC gyro observe the *same physical rotation*, so the **|gyro| envelope**
matches regardless of axis frame or bias. Resample both envelopes to a common grid, slide the (shorter)
OAK-D envelope across the FC envelope, and take the lag of peak normalized correlation. That lag is the
`t_mono → FC t_s` offset. Peak correlation is the alignment-quality metric.


In [ ]:
def _resample(t, x, dt, t0, t1):
    grid = np.arange(t0, t1, dt)
    return grid, np.interp(grid, t, x)

def align(oak_t, oak_env, fc_t, fc_env, dt=0.04):
    """Slide oak envelope over fc envelope; return (lag_s, peak_r, curve).

    lag_s maps OAK time to FC time:  fc_t ≈ oak_t + lag_s.
    """
    _, a = _resample(oak_t - oak_t[0], oak_env, dt, 0, oak_t[-1] - oak_t[0])
    _, b = _resample(fc_t  - fc_t[0],  fc_env,  dt, 0, fc_t[-1]  - fc_t[0])
    a = (a - a.mean()) / (a.std() + 1e-9)
    b = (b - b.mean()) / (b.std() + 1e-9)
    # normalized cross-correlation of template a within signal b
    corr = np.correlate(b, a, mode="valid")
    # per-position normalization by local energy of b (sliding window of length len(a))
    win = len(a)
    b2 = np.concatenate([[0.0], np.cumsum(b * b)])
    local_energy = b2[win:win + len(corr)] - b2[:len(corr)]
    denom = np.sqrt(np.maximum(local_energy, 1e-9)) * np.sqrt(np.dot(a, a))
    ncc = corr / (denom + 1e-9)
    k = int(np.argmax(ncc))
    lag_s = (fc_t[0] + k * dt) - oak_t[0]
    return lag_s, float(ncc[k]), ncc

for name in runs:
    r = runs[name]
    oak = r["imu"]; fc = r["fc"]["imu0"]
    lag, peak, ncc = align(oak["t_mono"].values, oak["g_mag"].values,
                           fc["t_s"].values,     fc["g_mag"].values)
    r["lag_s"], r["peak_r"] = lag, peak
    print(f"[{name}] best lag t_mono->FC t_s = {lag:8.2f}s   peak NCC = {peak:.3f}")

    fig, ax = plt.subplots(figsize=(11, 2.4))
    tt = fc["t_s"].values[0] + np.arange(len(ncc)) * 0.04     # FC t_s of OAK-window start
    ax.plot(tt, ncc, lw=0.8)
    ax.axvline(tt[int(np.argmax(ncc))], color="r", ls="--", lw=1)
    ax.set_title(f"{name}: normalized cross-correlation (peak {peak:.3f} @ lag {lag:.1f}s)")
    ax.set_xlabel("FC t_s of OAK window start"); ax.set_ylabel("NCC"); ax.grid(alpha=0.3)
    show(fig)


## 2. Compare the aligned IMUs

With the lag applied (`fc_t ≈ t_mono + lag`), overlay the OAK-D and FC |gyro| envelopes. Good overlap
confirms both the alignment and that the OAK-D IMU tracks the same rigid-body motion as the FC.


In [ ]:
for name in runs:
    r = runs[name]; oak = r["imu"]; fc = r["fc"]["imu0"]; lag = r["lag_s"]
    oak_t = oak["t_mono"].values + lag                       # now in FC t_s frame
    lo, hi = max(oak_t[0], fc["t_s"].min()), min(oak_t[-1], fc["t_s"].max())
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    fig.suptitle(f"{name}: OAK-D vs FC IMU0 (aligned, lag={lag:.1f}s, NCC={r['peak_r']:.2f})",
                 fontweight="bold", fontsize=10)
    axes[0].plot(oak_t, oak["g_mag"], lw=0.6, label="OAK-D |gyro|", color="steelblue")
    axes[0].plot(fc["t_s"], fc["g_mag"], lw=0.6, label="FC |gyro|", color="crimson", alpha=0.7)
    axes[0].set_ylabel("|gyro| (rad/s)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    axes[1].plot(oak_t, oak["a_mag"], lw=0.6, label="OAK-D |acc|", color="steelblue")
    axes[1].plot(fc["t_s"], fc["a_mag"], lw=0.6, label="FC |acc|", color="crimson", alpha=0.7)
    axes[1].axhline(9.81, color="gray", ls=":", lw=0.8)
    axes[1].set_ylabel("|acc| (m/s^2)"); axes[1].set_xlabel("FC t_s (s)")
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3); axes[1].set_xlim(lo, hi)
    show(fig)


## 3. Does the OAK-D IMU see the motor vibration?

Welch PSD of the OAK-D accelerometer, **handheld (motors off) vs armed (motors on)**, on the same axes.
The armed FC IMU hit **32.8 m/s² vibe with clipping**; the question for the hard-mount decision is whether
that vibration reaches the camera IMU. Remember the ~50 Hz Nyquist — a higher motor fundamental will alias
downward, so read *elevated broadband energy and any new peaks in armed*, not an exact motor frequency.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, comp, lbl in zip(axes, ["az", "g_mag"], ["accel Z", "|gyro|"]):
    for name, col in [("handheld", "gray"), ("armed", "crimson")]:
        r = runs[name]; x = r["imu"][comp].values - np.mean(r["imu"][comp].values)
        f, pxx = welch_psd(x, fs=r["fs_oak"], nperseg=2048)
        ax.semilogy(f, pxx, lw=1, color=col, label=f"{name} (fs={r['fs_oak']:.0f}Hz)")
    ax.set_title(f"OAK-D IMU PSD — {lbl}"); ax.set_xlabel("Hz")
    ax.set_ylabel("PSD"); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
show(fig)

# quantify the broadband ratio (armed/handheld) above 5 Hz
for comp in ["ax","ay","az"]:
    def band_power(name):
        r = runs[name]; x = r["imu"][comp].values - r["imu"][comp].mean()
        f, pxx = welch_psd(x, fs=r["fs_oak"], nperseg=2048)
        m = f > 5.0
        return np.trapezoid(pxx[m], f[m])
    print(f"{comp}: armed/handheld band power (>5 Hz) = {band_power('armed')/band_power('handheld'):6.1f}x")

# FC VIBE side-by-side for context
fig, ax = plt.subplots(figsize=(12, 3))
for name, col in [("handheld","gray"), ("armed","crimson")]:
    v = runs[name]["fc"]["F"].get("VIBE")
    if v is not None:
        v0 = v[v["IMU"] == 0]
        ax.plot(v0["t_s"], v0["VibeZ"], lw=0.8, color=col, label=f"{name} VibeZ")
ax.axhline(20, color="orange", ls="--", lw=1, label="20 m/s^2 threshold")
ax.set_title("FC VIBE (IMU0, Z)"); ax.set_xlabel("FC t_s (s)"); ax.set_ylabel("m/s^2")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
show(fig)


## 4. Feature-stream health

Matched-feature count per frame, handheld vs armed. A drop under armed motion/vibration (motion blur,
rolling-shutter smear, or IMU-driven mistracking) would itself degrade VINS independent of the IMU path.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
for name, col in [("handheld","gray"), ("armed","crimson")]:
    f = runs[name]["feat"]
    ax.plot(f["t_mono"] - f["t_mono"].iloc[0], f["n_feat"], lw=0.6, color=col,
            label=f"{name} (mean {f.n_feat.mean():.1f}, min {int(f.n_feat.min())})")
ax.set_title("chobits_features: matched-feature count / frame")
ax.set_xlabel("t since capture start (s)"); ax.set_ylabel("features")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
show(fig)


## 5. Plan — post-hoc pose production (the #35 estimator-half input-replayer)

The fixtures are the estimator **inputs**; to get pose we must run the *real* `vins_fusion` over them
offline. `harness/pose_replayer.py` today replays pose **output** into the router — we need the inverse.

**Build `harness/input_replayer.py` (new).**
1. `decode_fixture()` (this notebook) → interleave `chobits_imu` + `chobits_features` frames sorted by
   `t_mono` (their true send order); **re-send the raw payload bytes verbatim** (so the 13-vs-14 double
   question is moot — vins parses its own bytes).
2. `sendto` each datagram to `/tmp/chobits_imu` / `/tmp/chobits_features` where a **real
   `coordinator-vio-estimator`** container is bound. Pace by the inter-frame `t_mono` deltas (or a
   scale factor); run vins with **`multiple_thread: 0`** for deterministic replay.
3. `vins_fusion` emits pose on `/tmp/chobits_server` (10 floats: attitude + position + velocity).
   Tap with `vio-pose-tap` → **pose CSV** (reuse the router-half seam harness plumbing).

**Then, back in analysis:**
4. Align the regenerated VINS pose to the FC `XKF1`/`GPS` trajectory using the **same** `align()`
   lag found above (the fixture↔FC offset is already solved here — pose inherits the fixture clock).
5. **Compare the pair.** If VINS tracks GPS/EKF in *handheld* (vibration-free) but degrades in *armed*
   while the notch-filtered FC EKF stays good, that attributes the degradation to **motor vibration on the
   hard-mounted OAK-D** — the isolation decision. Section 3 above (does the camera IMU even see the
   vibration?) is the corroborating direct evidence.

Sequencing: this notebook does everything achievable from the inputs alone **today**; the pose curves drop
in once `input_replayer.py` exists. See #35 (harness/estimator-half), #42 (this analysis).


In [ ]:
summary = {name: dict(
    oak_imu_pkts=int(len(r["imu"])), oak_imu_hz=round(r["fs_oak"],1),
    feat_frames=int(len(r["feat"])), feat_mean=round(float(r["feat"].n_feat.mean()),1),
    fc_dur_s=round(r["fc"]["dur"],1),
    align_lag_s=round(r["lag_s"],2), align_ncc=round(r["peak_r"],3),
) for name, r in runs.items()}
print(json.dumps({"notebook": "vio-input-alignment", "runs": summary}, indent=2))
